# EFFResNet-ViT: Hybrid Deepfake Video Detection
### Implementation of IEEE SMART-2025 Paper
**Paper:** *Deepfake Video Detection: A Comprehensive Deep Learning-Based Approach* (Sharma & Sharma, 2025)

**Key Architecture Components:**
1. **EfficientNet-B0 Backbone**: Extracts local, high-frequency texture and boundary artifacts.
2. **ResNet-50 Backbone**: Extracts edge-based structural cues and facial contour representations.
3. **Locality-Preserving Fusion Neck**: Blends multi-scale feature maps with depthwise locality convolutions.
4. **Vision Transformer (ViT) Encoder**: Self-attention mechanism capturing global spatial correlations across facial regions.
5. **Locality Recovery & Classification Head**: Binary classification (0: Real, 1: Fake) with AUC-ROC tracking.


In [ ]:
# ============================================================
# 1. ENVIRONMENT & GPU HARDWARE VERIFICATION
# ============================================================
import os
import sys
import time
import json
import random
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights, ResNet50_Weights

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 65)
print('PyTorch Version :', torch.__version__)
print('Device          :', device)
if torch.cuda.is_available():
    print('GPU Device      :', torch.cuda.get_device_name(0))
    print('VRAM Available  :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
print('=' * 65)


In [ ]:
# ============================================================
# 2. DATASET DISCOVERY & MANIFEST BUILDER
# ============================================================
def locate_dataset_root() -> Path:
    candidate_dirs = [
        Path('/kaggle/input/faceforencispp-extracted-frames'),
        Path('/kaggle/input/datasets/adham7elmy/faceforencispp-extracted-frames'),
    ]
    for p in candidate_dirs:
        if p.exists() and (p / 'real').exists():
            return p
    
    # Dynamic fallback search under /kaggle/input
    kaggle_input = Path('/kaggle/input')
    for sub in kaggle_input.rglob('real'):
        if sub.is_dir() and (sub.parent / 'fake').exists():
            return sub.parent
            
    raise FileNotFoundError('FaceForensics++ dataset not found under /kaggle/input.')

DATASET_ROOT = locate_dataset_root()
print('Discovered Dataset Root:', DATASET_ROOT)
print('Top-level folders      :', [p.name for p in DATASET_ROOT.iterdir() if p.is_dir()])


In [ ]:
# ============================================================
# 3. GROUPED TRAIN / VALIDATION SPLIT (NO IDENTITY LEAKAGE)
# ============================================================
def build_ffpp_manifest(
    root: Path,
    max_frames_per_video: int = 15,
    val_split: float = 0.2,
    seed: int = 42
) -> Tuple[List[Tuple[str, int]], List[Tuple[str, int]]]:
    random.seed(seed)
    real_dir = root / 'real'
    fake_dir = root / 'fake'
    
    real_videos = {}
    if real_dir.exists():
        for vid in sorted(real_dir.iterdir()):
            if vid.is_dir():
                frames = sorted([str(p) for p in vid.glob('*.png')])
                if frames:
                    real_videos[vid.name] = frames
                    
    fake_videos = {}
    if fake_dir.exists():
        for method in sorted(fake_dir.iterdir()):
            if method.is_dir():
                for vid in sorted(method.iterdir()):
                    if vid.is_dir():
                        frames = sorted([str(p) for p in vid.glob('*.png')])
                        if frames:
                            fake_videos[f'{method.name}_{vid.name}'] = frames
                            
    print(f'Total Real Video Folders: {len(real_videos)}')
    print(f'Total Fake Video Folders: {len(fake_videos)}')
    
    real_keys = list(real_videos.keys())
    random.shuffle(real_keys)
    n_real_val = max(1, int(len(real_keys) * val_split))
    val_real_keys = set(real_keys[:n_real_val])
    train_real_keys = set(real_keys[n_real_val:])
    
    fake_keys = list(fake_videos.keys())
    random.shuffle(fake_keys)
    n_fake_val = max(1, int(len(fake_keys) * val_split))
    val_fake_keys = set(fake_keys[:n_fake_val])
    train_fake_keys = set(fake_keys[n_fake_val:])
    
    def sample_from_keys(keys, src, label):
        samples = []
        for k in keys:
            fl = src[k]
            if len(fl) <= max_frames_per_video:
                chosen = fl
            else:
                step = len(fl) / max_frames_per_video
                chosen = [fl[int(i * step)] for i in range(max_frames_per_video)]
            for f in chosen:
                samples.append((f, label))
        return samples
        
    tr_real = sample_from_keys(train_real_keys, real_videos, 0)
    va_real = sample_from_keys(val_real_keys, real_videos, 0)
    tr_fake = sample_from_keys(train_fake_keys, fake_videos, 1)
    va_fake = sample_from_keys(val_fake_keys, fake_videos, 1)
    
    # Balance classes equally
    n_tr = min(len(tr_real), len(tr_fake))
    train_samples = random.sample(tr_real, n_tr) + random.sample(tr_fake, n_tr)
    
    n_va = min(len(va_real), len(va_fake))
    val_samples = random.sample(va_real, n_va) + random.sample(va_fake, n_va)
    
    random.shuffle(train_samples)
    random.shuffle(val_samples)
    
    return train_samples, val_samples

train_samples, val_samples = build_ffpp_manifest(DATASET_ROOT, max_frames_per_video=12)
print(f'Final Balanced Train Samples: {len(train_samples):,} (Real: {len(train_samples)//2}, Fake: {len(train_samples)//2})')
print(f'Final Balanced Val Samples  : {len(val_samples):,} (Real: {len(val_samples)//2}, Fake: {len(val_samples)//2})')


In [ ]:
# ============================================================
# 4. PYTORCH DATASET & AUGMENTATION PIPELINE
# ============================================================
class FFPPFrameDataset(Dataset):
    def __init__(self, samples: List[Tuple[str, int]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform is not None:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)
            
        return img, torch.tensor(label, dtype=torch.float32)

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = FFPPFrameDataset(train_samples, transform=train_transforms)
val_dataset = FFPPFrameDataset(val_samples, transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
print('DataLoaders initialized successfully.')
print(f'Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}')


In [ ]:
# ============================================================
# 5. EFFResNet-ViT MODEL ARCHITECTURE DEFINITION
# ============================================================
class ConvLocalityBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.GELU()
        self.depthwise = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, padding=1, groups=out_channels, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()

    def forward(self, x):
        return self.act2(self.bn2(self.depthwise(self.act1(self.bn1(self.conv1(x))))))

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        normed = self.norm1(x)
        attn_out, _ = self.mha(normed, normed, normed)
        x = x + self.dropout1(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x

class EFFResNetViT(nn.Module):
    def __init__(
        self,
        embed_dim: int = 256,
        num_heads: int = 8,
        num_vit_layers: int = 4,
        mlp_ratio: float = 4.0,
        dropout: float = 0.1,
        pretrained: bool = True
    ):
        super().__init__()
        self.embed_dim = embed_dim
        
        # 1. EfficientNet-B0 (texture & fine artifacts)
        eff_weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        eff = models.efficientnet_b0(weights=eff_weights)
        self.eff_features = eff.features
        
        # 2. ResNet-50 (structural & edge cues)
        res_weights = ResNet50_Weights.DEFAULT if pretrained else None
        res = models.resnet50(weights=res_weights)
        self.res_stem = nn.Sequential(
            res.conv1, res.bn1, res.relu, res.maxpool,
            res.layer1, res.layer2, res.layer3, res.layer4
        )
        
        # 3. Hybrid Locality-Preserving Fusion Neck
        self.fusion_neck = ConvLocalityBlock(1280 + 2048, embed_dim)
        
        # 4. ViT Encoder
        self.num_patches = 7 * 7
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        
        self.transformer_layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_vit_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        
        # Post-attention locality refinement
        self.post_attn_locality = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1, groups=embed_dim, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.GELU()
        )
        
        self.feat_proj = nn.Linear(embed_dim * 2, embed_dim)
        
        # 5. Classifier
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
        
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def extract_features(self, x):
        B = x.size(0)
        f_eff = self.eff_features(x)
        f_res = self.res_stem(x)
        if f_eff.shape[2:] != f_res.shape[2:]:
            f_eff = F.interpolate(f_eff, size=f_res.shape[2:], mode='bilinear', align_corners=False)
        f_fused = self.fusion_neck(torch.cat([f_eff, f_res], dim=1))
        H, W = f_fused.shape[2], f_fused.shape[3]
        
        tokens = f_fused.flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_tokens, tokens], dim=1)
        tokens = self.pos_dropout(tokens + self.pos_embed[:, :tokens.size(1), :])
        
        for layer in self.transformer_layers:
            tokens = layer(tokens)
        tokens = self.norm(tokens)
        
        cls_out = tokens[:, 0]
        spatial_tokens = tokens[:, 1:]
        spatial_grid = spatial_tokens.transpose(1, 2).reshape(B, self.embed_dim, H, W)
        spatial_recovered = self.post_attn_locality(spatial_grid)
        spatial_pool = F.adaptive_avg_pool2d(spatial_recovered, (1, 1)).flatten(1)
        
        fused_vec = torch.cat([cls_out, spatial_pool], dim=1)
        return self.feat_proj(fused_vec)

    def forward(self, x):
        feat = self.extract_features(x)
        return self.classifier(feat)

model = EFFResNetViT(embed_dim=256, num_heads=8, num_vit_layers=4, pretrained=True).to(device)
print('Model initialized successfully.')
print(f'Total Model Parameters: {sum(p.numel() for p in model.parameters()):,} ({sum(p.numel() for p in model.parameters())/1e6:.2f}M)')


In [ ]:
# ============================================================
# 6. EVALUATION METRICS ENGINE
# ============================================================
def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred) * 100.0
    prec = precision_score(y_true, y_pred, zero_division=0) * 100.0
    rec = recall_score(y_true, y_pred, zero_division=0) * 100.0
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100.0
    try:
        auc = roc_auc_score(y_true, y_prob) * 100.0
    except ValueError:
        auc = 50.0
    cm = confusion_matrix(y_true, y_pred)
    return {
        'accuracy': round(acc, 2),
        'precision': round(prec, 2),
        'recall': round(rec, 2),
        'f1_score': round(f1, 2),
        'auc_roc': round(auc, 2),
        'confusion_matrix': cm.tolist()
    }


In [ ]:
# ============================================================
# 7. TRAINING & VALIDATION EXECUTION (WITH MIXED PRECISION)
# ============================================================
NUM_EPOCHS = 6
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

history = {
    'train_loss': [], 'val_loss': [],
    'val_acc': [], 'val_prec': [], 'val_rec': [], 'val_f1': [], 'val_auc': []
}
best_val_auc = 0.0

print('=' * 65)
print(f'Starting Training for {NUM_EPOCHS} Epochs on {device} (Mixed Precision: {device.type == "cuda"})')
print('=' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    t0 = time.time()
    
    for step, (imgs, lbls) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True).unsqueeze(1)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(imgs)
            loss = criterion(logits, lbls)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item()
        
        if (step + 1) % max(1, len(train_loader) // 4) == 0:
            print(f'Epoch [{epoch}/{NUM_EPOCHS}] Step [{step+1}/{len(train_loader)}] Loss: {loss.item():.4f}')
            
    scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Evaluation Phase
    model.eval()
    total_val_loss = 0.0
    all_lbls, all_probs = [], []
    
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            lbls_t = lbls.to(device, non_blocking=True).unsqueeze(1)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(imgs)
                loss = criterion(logits, lbls_t)
                probs = torch.sigmoid(logits)
                
            total_val_loss += loss.item()
            all_lbls.extend(lbls.numpy().tolist())
            all_probs.extend(probs.squeeze(1).cpu().numpy().tolist())
            
    avg_val_loss = total_val_loss / len(val_loader)
    metrics = compute_metrics(np.array(all_lbls), np.array(all_probs))
    epoch_time = time.time() - t0
    
    history['train_loss'].append(round(avg_train_loss, 4))
    history['val_loss'].append(round(avg_val_loss, 4))
    history['val_acc'].append(metrics['accuracy'])
    history['val_prec'].append(metrics['precision'])
    history['val_rec'].append(metrics['recall'])
    history['val_f1'].append(metrics['f1_score'])
    history['val_auc'].append(metrics['auc_roc'])
    
    print(f'>>> Epoch {epoch}/{NUM_EPOCHS} Complete ({epoch_time:.1f}s)')
    print(f'    Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')
    print(f'    Val Acc: {metrics["accuracy"]}% | Prec: {metrics["precision"]}% | Rec: {metrics["recall"]}%')
    print(f'    Val F1: {metrics["f1_score"]}% | Val AUC-ROC: {metrics["auc_roc"]}%')
    
    if metrics['auc_roc'] > best_val_auc:
        best_val_auc = metrics['auc_roc']
        ckpt_path = OUTPUT_DIR / 'best_effresnet_vit.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'metrics': metrics,
            'history': history
        }, ckpt_path)
        print(f'    [+] Saved new best model checkpoint to {ckpt_path.name} (AUC: {best_val_auc}%)')
        
print('=' * 65)
print(f'Training Finished! Best Validation AUC-ROC: {best_val_auc:.2f}%')
print('=' * 65)


In [ ]:
# ============================================================
# 8. RESULTS SUMMARY & BENCHMARK COMPARISON TABLE
# ============================================================
# Save metric history to JSON
with open(OUTPUT_DIR / 'metrics_summary.json', 'w') as f:
    json.dump({
        'best_val_auc': best_val_auc,
        'final_metrics': metrics,
        'history': history
    }, f, indent=2)

comparison_data = {
    'Model': ['XceptionNet (Paper Table I)', 'CViT (Paper Table I)', 'EFFResNet-ViT (Paper Table II)', 'EFFResNet-ViT (Our Reproduction)'],
    'Accuracy (%)': [96.3, 91.5, 98.4, metrics['accuracy']],
    'Precision (%)': ['-', '-', 97.2, metrics['precision']],
    'Recall (%)': ['-', '-', 98.6, metrics['recall']],
    'F1-Score (%)': ['-', '-', 97.6, metrics['f1_score']],
    'AUC-ROC (%)': [96.3, 91.0, 98.4, metrics['auc_roc']]
}

df_compare = pd.DataFrame(comparison_data)
print('\n' + '=' * 75)
print('BENCHMARK COMPARISON WITH PAPER TABLE I & TABLE II')
print('=' * 75)
print(df_compare.to_string(index=False))
print('=' * 75)

# Plot Training Curves
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, NUM_EPOCHS + 1), history['train_loss'], label='Train Loss', marker='o')
plt.plot(range(1, NUM_EPOCHS + 1), history['val_loss'], label='Val Loss', marker='s')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, NUM_EPOCHS + 1), history['val_acc'], label='Val Accuracy', marker='o')
plt.plot(range(1, NUM_EPOCHS + 1), history['val_auc'], label='Val AUC-ROC', marker='^')
plt.plot(range(1, NUM_EPOCHS + 1), history['val_f1'], label='Val F1', marker='x')
plt.title('Validation Metrics Progression')
plt.xlabel('Epoch')
plt.ylabel('Percentage (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=300)
plt.show()
print(f'Training plot saved to {OUTPUT_DIR / "training_curves.png"}')
